In [1]:
# %%
# ============================================================
# ☀️ HMI-Only Downloader → .NPZ Converter (Full-Disk, ±1 h)
# ============================================================
# • Downloads all 5 HMI products for each of 10 key flares
# • Uses ±1 h window around offsets (-24, -18, -12, -6, 0, +6)
# • Converts to 256×256 .npz and deletes FITS immediately
# • Skips any .npz already created  ✅ (resumable)
# ============================================================

import os, glob, cv2, numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from astropy.io import fits
import astropy.units as u
from sunpy.net import Fido, attrs as a

# ============================================================
# 🔧 Setup
# ============================================================

load_dotenv()
EMAIL = os.environ.get("EMAIL")
if not EMAIL:
    raise ValueError("❌ Missing EMAIL in .env — add EMAIL=your_registered_email@njit.edu")

BASE_DIR = os.path.abspath(".")
print(f"✅ Base directory: {BASE_DIR}")
print(f"✅ Using JSOC email: {EMAIL}")

# ============================================================
# ☀️ Final 10 Flares
# ============================================================

FLARES = [
    ("AR11158_M6.6","2011-02-13 17:28:00"),
    ("AR11158_X2.2","2011-02-15 01:44:00"),
    ("AR11261_M9.3","2011-07-30 02:04:00"),
    ("AR11429_X5.4","2012-03-07 00:02:00"),
    ("AR11429_M6.3","2012-03-09 03:22:00"),
    ("AR11520_X1.4","2012-07-12 15:37:00"),
    ("AR11719_M6.5","2013-04-11 06:55:00"),
    ("AR12036_M7.3","2014-04-18 12:31:00"),
    ("AR11944_X1.2","2014-01-07 18:04:00"),
    ("AR12017_X1.0","2014-03-29 17:35:00"),
]

# ============================================================
# 💾 FITS → NPZ converter + cleanup
# ============================================================

def fits_to_npz_and_cleanup(save_dir, out_npz, target_size=(256, 256)):
    """Combine all FITS in folder → .NPZ and delete FITS."""
    fits_files = glob.glob(os.path.join(save_dir, "*.fits"))
    if not fits_files:
        print(f"⚠️ No FITS in {save_dir}")
        return

    stacks = {}
    for fpath in fits_files:
        try:
            data = fits.getdata(fpath)
            if data is None:
                continue
            data = data.astype(np.float32)
            lo, hi = np.percentile(data, 1), np.percentile(data, 99)
            data = np.clip(data, lo, hi)
            data = np.log1p(data - lo + 1e-6)
            data = cv2.resize(data, target_size)

            # Identify HMI product
            ch = "unknown"
            if "hmi.B_720s" in fpath:
                if ".field" in fpath: ch = "hmiB_field"
                elif ".inclination" in fpath: ch = "hmiB_incl"
                elif ".azimuth" in fpath: ch = "hmiB_azim"
            elif "hmi.M_720s" in fpath: ch = "hmiM"
            elif "hmi.ic_720s" in fpath: ch = "hmiIC"

            stacks.setdefault(ch, []).append(data)
        except Exception as e:
            print(f"⚠️ Error reading {fpath}: {e}")

    if not stacks:
        print(f"⚠️ No valid data in {save_dir}")
        return

    packed = {k: np.stack(v, axis=0) for k, v in stacks.items() if len(v) > 0}
    np.savez_compressed(out_npz, **packed)
    print(f"💾 Saved {out_npz} ({len(packed)} channels)")

    # Cleanup
    for fpath in fits_files:
        try: os.remove(fpath)
        except: pass
    print(f"🧹 Deleted {len(fits_files)} FITS from {save_dir}")

# ============================================================
# 📥 HMI Downloader (resumable)
# ============================================================

def download_hmi_from_jsoc(start_time, end_time, save_dir, email):
    os.makedirs(save_dir, exist_ok=True)
    hmi_series = [
        ("hmi.B_720s", "field"),
        ("hmi.B_720s", "inclination"),
        ("hmi.B_720s", "azimuth"),
        ("hmi.M_720s", None),
        ("hmi.ic_720s", None),
    ]

    for series, seg in hmi_series:
        print(f"🔹 HMI {series} {seg or ''}: {start_time} → {end_time}")
        try:
            attrs = [
                a.Time(start_time, end_time),
                a.jsoc.Series(series),
                a.Sample(12*u.minute),
                a.jsoc.Notify(email),
            ]
            if seg:
                attrs.append(a.jsoc.Segment(seg))
            query = Fido.search(*attrs)
            if len(query) > 0:
                Fido.fetch(query, path=save_dir, max_conn=1, overwrite=False)
            else:
                print(f"⚠️ No data for {series} {seg or ''}")
        except Exception as e:
            print(f"⚠️ HMI {series} failed: {e}")

# ============================================================
# 🧭 Main Loop — Process ±1 h windows (resumable)
# ============================================================

for flare_name, flare_start in FLARES:
    flare_dir = os.path.join(BASE_DIR, flare_name, "full_disk", "npz_hmi")
    os.makedirs(flare_dir, exist_ok=True)

    flare_start_dt = datetime.strptime(flare_start, "%Y-%m-%d %H:%M:%S")
    offsets = [-24, -18, -12, -6, 0, 6]  # 6 timestamps per flare

    print(f"\n===============================================")
    print(f"☀️ Processing {flare_name} ({flare_start})")
    print("===============================================")

    for h in offsets:
        # ±1 hour window
        t0 = flare_start_dt + timedelta(hours=h - 0.5)
        t1 = flare_start_dt + timedelta(hours=h + 0.5)
        tag = t0.strftime("%Y%m%dT%H%M")
        save_dir = os.path.join(flare_dir, tag)
        os.makedirs(save_dir, exist_ok=True)
        out_npz = os.path.join(flare_dir, f"{tag}.npz")

        # Skip if npz already exists
        if os.path.exists(out_npz):
            print(f"⏭️ Skipping {flare_name}/{tag} — already exists")
            continue

        print(f"\n🕓 Download window: {t0} → {t1}")
        download_hmi_from_jsoc(
            start_time=t0.strftime("%Y-%m-%dT%H:%M:%S"),
            end_time=t1.strftime("%Y-%m-%dT%H:%M:%S"),
            save_dir=save_dir,
            email=EMAIL,
        )

        fits_to_npz_and_cleanup(save_dir, out_npz, target_size=(256, 256))

print("\n🎯 All HMI flares (±1 h) processed safely. Existing .npz skipped.")


/opt/miniconda3/envs/surya_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Base directory: /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data
✅ Using JSOC email: ss5369@njit.edu

☀️ Processing AR11158_M6.6 (2011-02-13 17:28:00)

🕓 Download window: 2011-02-12 16:58:00 → 2011-02-12 17:58:00
🔹 HMI hmi.B_720s field: 2011-02-12T16:58:00 → 2011-02-12T17:58:00


2025-11-02 18:48:41 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=2]
2025-11-02 18:48:41 - drms - INFO: Waiting for 0 seconds...
2025-11-02 18:48:42 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=1]
2025-11-02 18:48:42 - drms - INFO: Waiting for 5 seconds...
2025-11-02 18:48:47 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=1]
2025-11-02 18:48:47 - drms - INFO: Waiting for 5 seconds...
2025-11-02 18:48:53 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=1]
2025-11-02 18:48:53 - drms - INFO: Waiting for 5 seconds...
2025-11-02 18:49:04 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=1]
2025-11-02 18:49:04 - drms - INFO: Waiting for 5 seconds...
2025-11-02 18:49:11 - drms - INFO: Export request pending. [id=JSOC_20251102_005356, status=1]
2025-11-02 18:49:11 - drms - INFO: Waiting for 5 seconds...
2025-11-02 18:49:18 - sunpy - INFO: 6 URLs found for download. Full re

INFO: 6 URLs found for download. Full request totaling 126MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 6/6 [00:24<00:00,  4.13s/file]


...

🔹 HMI hmi.ic_720s : 2014-03-29T23:05:00 → 2014-03-30T00:05:00


2025-11-03 00:01:19 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=2]
2025-11-03 00:01:19 - drms - INFO: Waiting for 0 seconds...
2025-11-03 00:01:19 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=1]
2025-11-03 00:01:19 - drms - INFO: Waiting for 5 seconds...
2025-11-03 00:01:25 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=1]
2025-11-03 00:01:25 - drms - INFO: Waiting for 5 seconds...
2025-11-03 00:01:30 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=1]
2025-11-03 00:01:30 - drms - INFO: Waiting for 5 seconds...
2025-11-03 00:01:36 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=1]
2025-11-03 00:01:36 - drms - INFO: Waiting for 5 seconds...
2025-11-03 00:01:41 - drms - INFO: Export request pending. [id=JSOC_20251103_001472, status=1]
2025-11-03 00:01:41 - drms - INFO: Waiting for 5 seconds...
2025-11-03 00:01:46 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 6 URLs found for download. Full request totaling 87MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 6/6 [00:16<00:00,  2.80s/file]


💾 Saved /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data/AR12017_X1.0/full_disk/npz_hmi/20140329T2305.npz (2 channels)
🧹 Deleted 30 FITS from /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data/AR12017_X1.0/full_disk/npz_hmi/20140329T2305

🎯 All HMI flares (±1 h) processed safely. Existing .npz skipped.


In [2]:
# %%
# ============================================================
# 🔍 HMI Download Summary — Verify .NPZ Files and Channels
# ============================================================

import os, glob, numpy as np
from prettytable import PrettyTable

BASE_DIR = os.path.abspath(".")
flare_dirs = sorted([d for d in os.listdir(BASE_DIR) if d.startswith("AR")])

table = PrettyTable(["Flare", "Timestamp", "Num Channels", "Missing?", "Channels"])

expected_channels = {"hmiB_field", "hmiB_incl", "hmiB_azim", "hmiM", "hmiIC"}

for flare in flare_dirs:
    npz_files = sorted(glob.glob(os.path.join(BASE_DIR, flare, "full_disk", "npz_hmi", "*.npz")))
    if not npz_files:
        table.add_row([flare, "-", "-", "⚠️ No .npz found", "-"])
        continue

    for npz_path in npz_files:
        try:
            f = np.load(npz_path)
            keys = set(f.keys())
            missing = expected_channels - keys
            table.add_row([
                flare,
                os.path.basename(npz_path).replace(".npz", ""),
                len(keys),
                "⚠️ Missing " + ", ".join(missing) if missing else "✅ All 5",
                ", ".join(sorted(keys))
            ])
            f.close()
        except Exception as e:
            table.add_row([flare, os.path.basename(npz_path), "Error", str(e), "-"])

print(table)


+----------------+---------------+--------------+---------------------------------------------------+----------------+
|     Flare      |   Timestamp   | Num Channels |                      Missing?                     |    Channels    |
+----------------+---------------+--------------+---------------------------------------------------+----------------+
|  AR11158_M6.6  | 20110212T1658 |      2       | ⚠️ Missing hmiB_azim, hmiB_incl, hmiM, hmiB_field | hmiIC, unknown |
|  AR11158_M6.6  | 20110212T2258 |      2       | ⚠️ Missing hmiB_azim, hmiB_incl, hmiM, hmiB_field | hmiIC, unknown |
|  AR11158_M6.6  | 20110213T0458 |      2       | ⚠️ Missing hmiB_azim, hmiB_incl, hmiM, hmiB_field | hmiIC, unknown |
|  AR11158_M6.6  | 20110213T1058 |      2       | ⚠️ Missing hmiB_azim, hmiB_incl, hmiM, hmiB_field | hmiIC, unknown |
|  AR11158_M6.6  | 20110213T1658 |      2       | ⚠️ Missing hmiB_azim, hmiB_incl, hmiM, hmiB_field | hmiIC, unknown |
|  AR11158_M6.6  | 20110213T2258 |      2       

In [3]:
# %%
import os, glob, numpy as np
from prettytable import PrettyTable

# --- Define your 10 flares ---
FLARES = [
    "AR11158_M6.6",
    "AR11158_X2.2",
    "AR11261_M9.3",
    "AR11429_X5.4",
    "AR11429_M6.3",
    "AR11520_X1.4",
    "AR11719_M6.5",
    "AR12036_M7.3",
    "AR11944_X1.2",
    "AR12017_X1.0",
]

BASE_DIR = os.path.abspath(".")
table = PrettyTable(["Flare", "Timestamp", "Has_hmiIC", "Has_unknown", "Num Channels", "Channels"])

missing_summary = {f: 0 for f in FLARES}

# --- Scan through each flare ---
for flare in FLARES:
    npz_dirs = glob.glob(os.path.join(BASE_DIR, flare, "full_disk", "npz_hmi", "*.npz"))
    if not npz_dirs:
        table.add_row([flare, "-", "-", "-", "-", "⚠️ No .npz found"])
        continue

    for npz_path in sorted(npz_dirs):
        data = np.load(npz_path)
        keys = list(data.keys())
        has_ic = "hmiIC" in keys
        has_unk = "unknown" in keys
        if not (has_ic and has_unk):
            missing_summary[flare] += 1
        table.add_row([
            flare,
            os.path.basename(npz_path).replace(".npz",""),
            "✅" if has_ic else "❌",
            "✅" if has_unk else "❌",
            len(keys),
            ", ".join(keys)
        ])
        data.close()

print(table)

print("\n=== Summary of Missing HMI Channels ===")
for f, miss in missing_summary.items():
    print(f"{f:<15} → {miss} window(s) missing one or both HMI channels")


+--------------+---------------+-----------+-------------+--------------+----------------+
|    Flare     |   Timestamp   | Has_hmiIC | Has_unknown | Num Channels |    Channels    |
+--------------+---------------+-----------+-------------+--------------+----------------+
| AR11158_M6.6 | 20110212T1658 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110212T2258 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_M6.6 | 20110213T0458 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110213T1058 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110213T1658 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_M6.6 | 20110213T2258 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 | 20110214T0114 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 | 20110214T0714 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 

In [4]:
# %%
import os, glob, numpy as np
from prettytable import PrettyTable

# --- Your 10 final flares ---
FLARES = [
    "AR11158_M6.6",
    "AR11158_X2.2",
    "AR11261_M9.3",
    "AR11429_X5.4",
    "AR11429_M6.3",
    "AR11520_X1.4",
    "AR11719_M6.5",
    "AR12036_M7.3",
    "AR11944_X1.2",
    "AR12017_X1.0",
]

BASE_DIR = os.path.abspath(".")
table = PrettyTable(["Flare", "Timestamp", "Has_hmiIC", "Has_unknown", "Num Channels", "Channels"])
summary = []

# --- Check each flare ---
for flare in FLARES:
    npz_paths = sorted(glob.glob(os.path.join(BASE_DIR, flare, "full_disk", "npz_hmi", "*.npz")))
    num_windows = len(npz_paths)
    if num_windows == 0:
        table.add_row([flare, "-", "-", "-", "-", "⚠️ No .npz found"])
        summary.append((flare, 0, 0))
        continue

    missing_count = 0
    for npz_path in npz_paths:
        data = np.load(npz_path)
        keys = list(data.keys())
        has_ic = "hmiIC" in keys
        has_unk = "unknown" in keys
        if not (has_ic and has_unk):
            missing_count += 1
        table.add_row([
            flare,
            os.path.basename(npz_path).replace(".npz", ""),
            "✅" if has_ic else "❌",
            "✅" if has_unk else "❌",
            len(keys),
            ", ".join(keys)
        ])
        data.close()
    summary.append((flare, num_windows, missing_count))

# --- Display detailed table ---
print(table)

# --- Summary Section ---
print("\n=== 📊 Summary per Flare ===")
for flare, total, missing in summary:
    if total == 0:
        print(f"{flare:<15} → ⚠️ No NPZ files found")
    else:
        print(f"{flare:<15} → {total} total windows | {missing} missing hmiIC or unknown")

print("\n✅ Done. Each flare ideally should have 6 NPZ windows (−24, −18, −12, −6, 0, +6 hours).")


+--------------+---------------+-----------+-------------+--------------+----------------+
|    Flare     |   Timestamp   | Has_hmiIC | Has_unknown | Num Channels |    Channels    |
+--------------+---------------+-----------+-------------+--------------+----------------+
| AR11158_M6.6 | 20110212T1658 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110212T2258 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_M6.6 | 20110213T0458 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110213T1058 |     ✅    |      ✅     |      2       | hmiIC, unknown |
| AR11158_M6.6 | 20110213T1658 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_M6.6 | 20110213T2258 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 | 20110214T0114 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 | 20110214T0714 |     ✅    |      ✅     |      2       | unknown, hmiIC |
| AR11158_X2.2 